In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns
import time

T1 = time.time()

# 指定分号为字段分隔符
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/4-2/03870.csv', sep=';') 
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/M3353_gpsdistance_normalized(2).csv') #20条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/M3353_gpsdistance_normalized(2).csv') #10条
fdata = pd.read_csv('E:/大数据/毕业设计/数据/datas/40/M2333_gpsdistance_normalized.csv') #10
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/datas/20/M2503_gpsdistance_normalized.csv') #10条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/datas/10/M5583_gpsdistance_normalized.csv') #20条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/4-2/M2333.csv', sep=';')  #40条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/result/M4123_dropdata.csv')   #10条
display(fdata)

,idx,opath,lineName,direction,t_flag,time,lng,lat,distance_of_gpspoint,gps_normalized_distance_length
0,BS04317D,63083,M2333,1.0,1,2019-01-04 07:03:27,113.906742,22.709049,0.432755,0.010337
1,BS04317D,63083,M2333,1.0,1,2019-01-04 07:36:46,113.906392,22.709334,0.480602,0.011480
2,BS04317D,63083,M2333,1.0,1,2019-01-04 07:36:56,113.906379,22.709346,0.482525,0.011526
3,BS04317D,63083,M2333,1.0,1,2019-01-04 07:37:06,113.906344,22.709380,0.487659,0.011649
4,BS04317D,63083,M2333,1.0,1,2019-01-04 07:47:36,113.906416,22.709311,0.477055,0.011396
...,...,...,...,...,...,...,...,...,...,...
130962,BS08123D,75772,M2333,2.0,3,2019-01-04 16:05:15,113.904974,22.710708,41.662506,0.966983
130963,BS08123D,75771,M2333,2.0,3,2019-01-04 16:05:25,113.906310,22.709231,41.892125,0.972313
130964,BS08123D,75771,M2333,2.0,3,2019-01-04 16:05:35,113.906427,22.709129,41.908692,0.972697
130965,BS08123D,75771,M2333,2.0,3,2019-01-04 16:17:45,113.906521,22.709055,41.921378,0.972992


In [2]:
import pandas as pd

# 定义 assign_to_windows 函数
def assign_to_windows(row):
    current_time = row['time']
    minutes_since_midnight = (current_time - current_time.normalize()).total_seconds() / 60.0
    
    # 计算基本时间窗口的开始时间，确保每个窗口覆盖4分钟
    window_start_minute = int(minutes_since_midnight // 4 * 4)  # 使用整除并乘以4确保步进为4
    window_end_minute = window_start_minute + 4  # 窗口结束时间调整为开始时间后的4分钟
    
    return pd.Series([window_start_minute, window_end_minute])

# 将 'time' 列的数据类型从字符串转换为 datetime
fdata['time'] = pd.to_datetime(fdata['time'])

# 应用函数，并创建新的列来存储分配结果
fdata[['win_st', 'win_en']] = fdata.apply(assign_to_windows, axis=1)

In [3]:
# 基于车牌号（'idx'列）、线路号（'linenumber'列）和时间窗口的开始时间（'win_st'列）对数据进行分组
counts_per_group = fdata.groupby(['lineName', 'idx', 'win_st']).size()

# 过滤掉那些在同一时间窗口内每组只有一条记录的数据
# 这会返回一个包含每组在每个时间窗口内记录数大于1的过滤后的DataFrame的索引
filtered_idx = counts_per_group[counts_per_group > 1].index

# 基于过滤后的索引，我们选择原始数据集中符合条件的行
# 这里的'linenumber', 'idx'和'win_st'必须与groupby中使用的列名完全匹配
fdata_filtered = fdata[fdata.set_index(['lineName', 'idx', 'win_st']).index.isin(filtered_idx)]

display(fdata_filtered)

,idx,opath,lineName,direction,t_flag,time,lng,lat,distance_of_gpspoint,gps_normalized_distance_length,win_st,win_en
1,BS04317D,63083,M2333,1.0,1,2019-01-04 07:36:46,113.906392,22.709334,0.480602,0.011480,456,460
2,BS04317D,63083,M2333,1.0,1,2019-01-04 07:36:56,113.906379,22.709346,0.482525,0.011526,456,460
3,BS04317D,63083,M2333,1.0,1,2019-01-04 07:37:06,113.906344,22.709380,0.487659,0.011649,456,460
4,BS04317D,63083,M2333,1.0,1,2019-01-04 07:47:36,113.906416,22.709311,0.477055,0.011396,464,468
5,BS04317D,63083,M2333,1.0,1,2019-01-04 07:47:46,113.906424,22.709304,0.475932,0.011369,464,468
...,...,...,...,...,...,...,...,...,...,...,...,...
130962,BS08123D,75772,M2333,2.0,3,2019-01-04 16:05:15,113.904974,22.710708,41.662506,0.966983,964,968
130963,BS08123D,75771,M2333,2.0,3,2019-01-04 16:05:25,113.906310,22.709231,41.892125,0.972313,964,968
130964,BS08123D,75771,M2333,2.0,3,2019-01-04 16:05:35,113.906427,22.709129,41.908692,0.972697,964,968
130965,BS08123D,75771,M2333,2.0,3,2019-01-04 16:17:45,113.906521,22.709055,41.921378,0.972992,976,980


In [4]:
# 计算角度
def calculate_right_angle_with_horizontal(end_dis, start_dis):
    angle = np.degrees(np.arctan2(end_dis - start_dis, 1)) % 360
    return angle

# 获取每辆车的首尾位置
start_dis = fdata_filtered.groupby(['idx', 'win_st'])['gps_normalized_distance_length'].first().reset_index(name='start_dis')
end_dis = fdata_filtered.groupby(['idx', 'win_st'])['gps_normalized_distance_length'].last().reset_index(name='end_dis')

# 合并数据并计算角度
dis_positions = pd.merge(start_dis, end_dis, on=['idx', 'win_st'])
dis_positions['angle'] = dis_positions.apply(lambda row: calculate_right_angle_with_horizontal(row['end_dis'], row['start_dis']), axis=1)

# 将计算好的角度合并回 fdata_filtered
fdata_filtered = pd.merge(fdata_filtered, dis_positions[['idx', 'win_st', 'angle']], on=['idx', 'win_st'])

# 计算MBR
def calculate_mbrs(group):
    min_time = group['time'].min()
    max_time = group['time'].max()
    min_dis_location = group['gps_normalized_distance_length'].min()
    max_dis_location = group['gps_normalized_distance_length'].max()
    return pd.Series([min_time, max_time, min_dis_location, max_dis_location], index=['min_time', 'max_time', 'min_dis_location', 'max_dis_location'])

# 辅助函数，计算两点确定的直线的参数（斜率和截距）
def calculate_line_parameters(point1, point2):
    x_coords, y_coords = [point1[0], point2[0]], [point1[1], point2[1]]
    A = np.vstack([x_coords, np.ones(len(x_coords))]).T
    m, c = np.linalg.lstsq(A, y_coords, rcond=None)[0]
    return m, c

# 辅助函数，计算点到直线的距离
def point_to_line_distance(point, line_points):
    x0, y0 = point
    x1, y1 = line_points[0]
    x2, y2 = line_points[1]
    m, c = calculate_line_parameters((x1, y1), (x2, y2))
    A, B, C = -m, 1, -c
    distance = abs(A * x0 + B * y0 + C) / (np.sqrt(A**2 + B**2))
    return distance

# 函数，计算两条线段之间的最大垂直距离
def calculate_max_vertical_distance_between_lines(line1_points, line2_points):
    point1_line2, point2_line2, midpoint_line2 = line2_points
    distance_start = point_to_line_distance(point1_line2, line1_points)
    distance_mid = point_to_line_distance(midpoint_line2, line1_points)
    distance_end = point_to_line_distance(point2_line2, line1_points)
    return max(distance_start, distance_mid, distance_end)

# 检查MBR是否有交集
def mbrs_intersect(mbr1, mbr2):
    return not (mbr1['max_time'] < mbr2['min_time'] or mbr1['min_time'] > mbr2['max_time'] or
                mbr1['max_dis_location'] < mbr2['min_dis_location'] or mbr1['min_dis_location'] > mbr2['max_dis_location'])

# 检测公交车串车的函数
def detect_bus_bunching(data, threshold):
    bunching_events = []
    
    for (lineName, win_st, direction), window_data in data.groupby(['lineName', 'win_st', 'direction']):
        # 获取每辆车的首尾位置和角度
        lines = window_data.groupby('idx').agg({
            'gps_normalized_distance_length': ['first', 'last'], 
            'time': ['first', 'last'],
            'angle': 'first'
        }).reset_index()

        lines.columns = ['idx', 'start_dis', 'end_dis', 'start_time', 'end_time', 'angle']
        
        lines['start_time'] = lines['start_time'].apply(lambda x: x.timestamp())
        lines['end_time'] = lines['end_time'].apply(lambda x: x.timestamp())
        
        # 按角度排序
        lines = lines.sort_values(by='angle').reset_index(drop=True)

        # 计算每对车辆之间的最大垂直距离并检查MBR
        for i in range(len(lines) - 1):
            angle_diff = abs(lines.at[i + 1, 'angle'] - lines.at[i, 'angle'])
            if angle_diff <= 3:
                mbr1 = calculate_mbrs(window_data[window_data['idx'] == lines.at[i, 'idx']])
                mbr2 = calculate_mbrs(window_data[window_data['idx'] == lines.at[i + 1, 'idx']])
                
                if mbrs_intersect(mbr1, mbr2):
                    line1_points = [(lines.at[i, 'start_time'], lines.at[i, 'start_dis']), 
                                    (lines.at[i, 'end_time'], lines.at[i, 'end_dis'])]
                    line2_points = [(lines.at[i + 1, 'start_time'], lines.at[i + 1, 'start_dis']), 
                                    (lines.at[i + 1, 'end_time'], lines.at[i + 1, 'end_dis'])]
                    midpoint_line2 = ((line2_points[0][0] + line2_points[1][0]) / 2, 
                                      (line2_points[0][1] + line2_points[1][1]) / 2)
                    distance = calculate_max_vertical_distance_between_lines(line1_points, [line2_points[0], midpoint_line2, line2_points[1]])
                    T2 = time.time()
                    print('程序运行时间:%s秒' % ((T2 - T1)*1))
                    # 计算串车持续时间
                    min_time = min(mbr1['min_time'], mbr2['min_time'])
                    max_time = max(mbr1['max_time'], mbr2['max_time'])
                    duration = max_time - min_time

                    if distance < threshold:
                        bunching_events.append({
                            'lineName': lineName,
                            'direction': direction,
                            'idx1': lines.at[i, 'idx'],
                            'idx2': lines.at[i + 1, 'idx'],
                            'win_st': win_st,
                            'distance': distance,
                            'start_time': min_time,
                            'end_time': max_time,
                            'duration': duration
                        })

    return pd.DataFrame(bunching_events)

# 检测公交车串车
threshold = 0.01
bunching_df = detect_bus_bunching(fdata_filtered, threshold)

def merge_bunching_events(events):
    events = events.sort_values(by='win_st').reset_index(drop=True)
    merged_events = []
    i = 0

    while i < len(events):
        current_event = events.iloc[i]
        j = i + 1

        while j < len(events):
            next_event = events.iloc[j]
            if (current_event['idx1'] == next_event['idx1'] and current_event['idx2'] == next_event['idx2'] and
                (next_event['start_time'] - current_event['end_time']).total_seconds() < 5):
                current_event['end_time'] = next_event['end_time']
                current_event['duration'] = current_event['end_time'] - current_event['start_time']
                current_event['win_st'] = f"{current_event['win_st']},{next_event['win_st']}"
                j += 1
            else:
                break

        merged_events.append(current_event)
        i = j

    merged_df = pd.DataFrame(merged_events)
    # 过滤掉持续时间小于3分钟的事件
    filtered_merged_df = merged_df[merged_df['duration'] >= timedelta(minutes=3)]
    return filtered_merged_df

# 合并串车事件
merged_bunching_df = merge_bunching_events(bunching_df)

# 输出合并后的串车事件
print(merged_bunching_df)

程序运行时间:17.45958971977234秒
程序运行时间:17.477601766586304秒
程序运行时间:17.560635328292847秒
程序运行时间:17.6136474609375秒
程序运行时间:17.641654014587402秒
程序运行时间:17.67464780807495秒
程序运行时间:17.6806480884552秒
程序运行时间:17.708653450012207秒
程序运行时间:17.71266484260559秒
程序运行时间:17.716668605804443秒
程序运行时间:17.75066328048706秒
程序运行时间:17.8346951007843秒
程序运行时间:17.868699073791504秒
程序运行时间:17.935704231262207秒
程序运行时间:18.096755981445312秒
程序运行时间:18.13976812362671秒
程序运行时间:18.166768789291382秒
程序运行时间:18.179771661758423秒
程序运行时间:18.186778783798218秒
程序运行时间:18.199772357940674秒
程序运行时间:18.218769073486328秒
程序运行时间:18.23177146911621秒
程序运行时间:18.338804244995117秒
程序运行时间:18.379819631576538秒
程序运行时间:18.4188129901886秒
程序运行时间:18.456836223602295秒
程序运行时间:18.49784541130066秒
程序运行时间:18.501846075057983秒
程序运行时间:18.549850702285767秒
程序运行时间:18.58286428451538秒
程序运行时间:18.587865829467773秒
程序运行时间:18.607906579971313秒
程序运行时间:18.63590693473816秒
程序运行时间:18.65090298652649秒
程序运行时间:18.691910982131958秒
程序运行时间:18.71992540359497秒
程序运行时间:18.73592185974121秒
程序运行时间:18.73792219161

程序运行时间:24.9655544757843秒
程序运行时间:25.000874042510986秒
程序运行时间:25.045884609222412秒
程序运行时间:25.082830905914307秒
程序运行时间:25.234825134277344秒
程序运行时间:25.27480983734131秒
程序运行时间:25.44985032081604秒
程序运行时间:25.493858814239502秒
程序运行时间:25.582886934280396秒
程序运行时间:25.621896266937256秒
程序运行时间:25.636886596679688秒
程序运行时间:25.660905122756958秒
程序运行时间:25.676908016204834秒
程序运行时间:25.752917051315308秒
程序运行时间:25.7849178314209秒
程序运行时间:25.9339599609375秒
程序运行时间:25.938961505889893秒
程序运行时间:25.99197793006897秒
程序运行时间:26.014984130859375秒
程序运行时间:26.041620016098022秒
程序运行时间:26.04361915588379秒
程序运行时间:26.062626838684082秒
程序运行时间:26.093635082244873秒
程序运行时间:26.096636056900024秒
程序运行时间:26.161632537841797秒
程序运行时间:26.173650979995728秒
程序运行时间:26.17565083503723秒
程序运行时间:26.209656953811646秒
程序运行时间:26.221251487731934秒
程序运行时间:26.223251819610596秒
程序运行时间:26.277796506881714秒
程序运行时间:26.31280493736267秒
程序运行时间:26.3278067111969秒
程序运行时间:26.329809188842773秒
程序运行时间:26.37881875038147秒
程序运行时间:26.420828342437744秒
程序运行时间:26.427830696105957秒
程序运行时间:26.442832

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17052\3114642888.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_event['end_time'] = next_event['end_time']
C:\Users\ASUS\AppData\Local\Temp\ipykernel_17052\3114642888.py:130: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_event['duration'] = current_event['end_time'] - current_event['start_time']
C:\Users\ASUS\AppData\Local\Temp\ipykernel_17052\3114642888.py:131: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.